# Homework 2 - Xinlei Cheng

In [0]:
from pyspark.sql.functions import (
    col, avg, min, max, rank, row_number, count, sum,
    when, current_date, floor, months_between, datediff,
    to_date, concat, lit, upper, substring, coalesce,
    first, last, dense_rank, struct
)
from pyspark.sql.window import Window

In [0]:
df_pit_stops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)

### Question 1: What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

In [0]:
df_pit_stops.head(10)

In [0]:
df_pit = df_pit_stops.withColumn("milliseconds", col("milliseconds").cast("int"))

# Average pit stop time per driver per race
df_avg_pit_per_driver = (
    df_pit
    .groupBy("raceId", "driverId")
    .agg(
        avg("milliseconds").alias("avg_pit_time_ms")
    )
)

display(df_avg_pit_per_driver)


In [0]:
# Fastest and slowest pit stop per race
df_race_pit_extremes = (
    df_pit
    .groupBy("raceId")
    .agg(
        min(col("milliseconds")) .alias("fastest_pit_stop_ms"),
        max(col("milliseconds")).alias("slowest_pit_stop_ms")
    )
)

# Combine them into one
df_q1 = (
    df_avg_pit_per_driver
    .join(df_race_pit_extremes, on="raceId", how="inner")
    .orderBy("raceId", "driverId")
)

display(df_q1)